In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Types")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=" * 70)
print("STEP 1: Job parameters")
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  merge_keys        : '{merge_keys}'")
print(f"  load_type         : '{load_type}'")
print("=" * 70)

In [0]:
source_table = f"adb_caresync.bronze.{table_name}"
filter_condition = f"landing_timestamp = '{landing_timestamp}'"

print("STEP 2: Reading bronze data")
print(f"  Source table : {source_table}")
print(f"  Filter       : {filter_condition}")
print(f"  Query        : SELECT * FROM {source_table} WHERE {filter_condition}")

bronze_df = spark.read.table(source_table).filter(filter_condition)

print(f"  Rows read    : {bronze_df.count()}")
print(f"  Columns      : {bronze_df.columns}")
print("-" * 70)

In [0]:
from pyspark.sql import functions as F
print("STEP 3: Adding audit columns")
print("  insert_timestamp = current_timestamp()")
print("  update_timestamp = current_timestamp()")

updated_df=bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn("update_timestamp", F.current_timestamp())

print(f"  Final columns : {updated_df.columns}")
print("-" * 70)

In [0]:
from delta.tables import DeltaTable

target_table = f"adb_caresync.silver.{table_name}"
# Empty load_type defaults to FULL to keep the previous behaviour
load_type_norm = (load_type or "FULL").strip().upper()

print("STEP 4: Writing to silver")
print(f"  Target table      : {target_table}")
print(f"  Requested load    : '{load_type}' -> resolved as {load_type_norm}")
print(f"  Target exists     : {spark.catalog.tableExists(target_table)}")

if load_type_norm == "FULL":
    print("  Action            : Overwrite target table with source data")
    print(f"  Query equivalent  : CREATE OR REPLACE TABLE {target_table} AS SELECT * FROM <source_df>")
    updated_df.write.mode("overwrite").saveAsTable(target_table)

elif load_type_norm == "APPEND":
    print("  Action            : Append source data to target table")
    print(f"  Query equivalent  : INSERT INTO {target_table} SELECT * FROM <source_df>")
    updated_df.write.mode("append").saveAsTable(target_table)

elif load_type_norm == "MERGE":
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type = 'MERGE'")
    missing = [k for k in keys if k not in updated_df.columns]
    if missing:
        raise ValueError(f"Merge keys not found in source data: {missing}")

    print(f"  Merge keys        : {keys}")

    # Keep one row per key so the MERGE does not fail on multiple source matches
    source_df = updated_df.dropDuplicates(keys)
    src_count, dedup_count = updated_df.count(), source_df.count()
    print(f"  Source rows       : {src_count} (after de-dup on keys: {dedup_count}, dropped: {src_count - dedup_count})")

    if not spark.catalog.tableExists(target_table):
        # First run: create the target table
        print("  Action            : Target not found - creating it from source data")
        print(f"  Query equivalent  : CREATE TABLE {target_table} AS SELECT * FROM <source_df>")
        source_df.write.mode("overwrite").saveAsTable(target_table)
    else:
        merge_condition = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
        # On update keep the original insert_timestamp; refresh everything else
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"}
        insert_values = {f"`{c}`": f"s.`{c}`" for c in source_df.columns}

        print("  Action            : MERGE (update matched, insert new)")
        print(f"  Merge condition   : {merge_condition}")
        print(f"  Columns updated   : {[c for c in source_df.columns if c != 'insert_timestamp']}")
        print("  Column preserved  : insert_timestamp (kept from target on update)")

        set_clause = ",\n      ".join([f"t.{k} = {v}" for k, v in update_set.items()])
        insert_cols = ", ".join(insert_values.keys())
        insert_vals = ", ".join(insert_values.values())
        merge_sql = (
            f"MERGE INTO {target_table} AS t\n"
            f"USING <source_df> AS s\n"
            f"ON {merge_condition}\n"
            f"WHEN MATCHED THEN UPDATE SET\n      {set_clause}\n"
            f"WHEN NOT MATCHED THEN INSERT ({insert_cols})\n"
            f"  VALUES ({insert_vals})"
        )
        print("  Query equivalent  :")
        print(merge_sql)

        (DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_values)
            .execute())

else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected one of: FULL, APPEND, MERGE")

# Report what the write actually did, from the Delta transaction log
last_op = DeltaTable.forName(spark, target_table).history(1).select("version", "operation", "operationMetrics").first()
print("-" * 70)
print("STEP 5: Write result")
print(f"  Delta version     : {last_op['version']}")
print(f"  Operation         : {last_op['operation']}")
for k, v in sorted((last_op["operationMetrics"] or {}).items()):
    if k.startswith("num"):
        print(f"  {k:<28}: {v}")
print(f"  Target row count  : {spark.read.table(target_table).count()}")
print("=" * 70)
print(f"{load_type_norm} load completed for {target_table}")

In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))